# RAG 基础 — LangGraph + Chroma

## 2026-05-15 | 第2周周五 | embedding / chunking / 混合检索

今日目标：用 Chroma 建一个最小知识库，把检索封装成 `retrieve_context` 工具，再接入 LangGraph Agent。

RAG pipeline：`Indexing → Retrieval → Augmentation → Generation`。

- Indexing：文档清洗、chunking、embedding、写入向量库
- Retrieval：用户问题向量化，召回 top-k chunk
- Augmentation：把证据注入 prompt
- Generation：LLM 基于证据回答并给引用

In [ ]:
from agent import build_collection, run_self_test

collection = build_collection(reset=True)
run_self_test(collection)

上面的离线自测验证两件事：

1. Chroma collection 能正常写入和查询。
2. 当前实现使用向量距离 + 关键词分数融合，演示混合检索的基本思想。

In [ ]:
from agent import build_graph

app = build_graph(collection)
print(app.get_graph(xray=True).draw_mermaid())

In [ ]:
from agent import run_agent_demo

# 需要 .env 中配置 API_KEY
run_agent_demo(collection)

面试速记：

- chunk 太大：噪声多、占 context、容易 Lost in the Middle。
- chunk 太小：语义断裂、因果链不完整。
- 混合检索：BM25/关键词补精确匹配，embedding 补语义泛化。
- reranker：粗召回后做二阶段精排，提高 grounding，但增加延迟成本。
- Agent 记忆：短期记忆在 state/history，长期记忆在向量库/数据库/文件系统。